# Thinning

In [ ]:
import imageio
from matplotlib import pyplot as plt
import numpy as np
from scipy.ndimage import binary_dilation, binary_erosion, binary_hit_or_miss

import morphological_processing_sol


def diff_check(img1, img2):
    img1 = img1.astype(bool)
    img2 = img2.astype(bool)

    num_diff = np.sum(img1 != img2)
    if num_diff > 1:
        return False
    else:
        return True

: 

Write a function `morpho_thin(I, B, numiter)` for thinning and returning the foreground of binary image `I` using a sequence of $K$, $3\times 3$ structuring elements contained in $3\times 3\times K$ array, `B`.  The function stops when `numiter` of complete iterations through all $K$ structuring elements is reached.  

In [ ]:
### helper functions
def morpho_erode(I, B):
    """
    Perform morphological erosion on a binary image.

    Parameters
    ----------
    image : numpy.ndarray
        The input binary image as a 2D NumPy array.
    structure : numpy.ndarray, optional
        The structuring element used for erosion. If None, a cross-shaped structuring element is used (default is None).
    iterations : int, optional
        The number of times erosion is applied (default is 1).

    Returns
    -------
    numpy.ndarray
        The eroded binary image as a 2D NumPy array.
    """

    # YOUR CODE HERE
    #get dimensions of image and structuring element
    M,N = I.shape
    p,q = B.shape
    B_sum = np.sum(B)
    # initialize output image
    im_out = np.zeros_like(I)
    # pad image, start with zero padding
    im_pad = np.pad(I,((p//2,p//2),(q//2,q//2)))
    # iterate through image
    for i in range(M):
        for j in range(N):
            #find the neighborhood of interest
            hood = im_pad[i:i+p, j:j+q]
            # determine output pixel
            # mask the area of interest with the structure, then compare their sums
            mask = np.sum(hood*B)
            if mask == B_sum:
                im_out[i,j] = 1
            # redundant code, we initialized im_out to all 0s
            # else:
            #     im_out[i,j] = 0
    return im_out


def morpho_dilate(I, B):
    """
    Perform morphological dilation on a binary image.

    Parameters
    ----------
    image : numpy.ndarray
        The input binary image as a 2D NumPy array.
    structure : numpy.ndarray, optional
        The structuring element used for dilation. If None, a cross-shaped structuring element is used (default is None).
    iterations : int, optional
        The number of times dilation is applied (default is 1).

    Returns
    -------
    numpy.ndarray
        The dilated binary image as a 2D NumPy array.
    """

    # YOUR CODE HERE
    M,N = I.shape
    p,q = B.shape
    # initialize output image
    im_out = np.zeros_like(I)
    # pad image
    im_pad = np.pad(I,((p//2,p//2),(q//2,q//2)))
    #iterate through image
    for i in range(M):
        for j in range(N):
            # get area of interest
            hood = im_pad[i:i+p, j:j+q]
            # determine output pixel
            if np.sum(hood*B) > 0:
                im_out[i,j] = 1
    return im_out

def hit_or_miss(I,B):
    # Separate foreground and background requirements
    B1 = (B == 1) # foreground pixels
    B0 = (B == 0) # background pixels
    I_comp = np.logical_not(I)

    # Erode original image using foreground requirements
    fg_erode = morpho_erode(I,B1)

    # Erode complement using background requirements
    bg_erode = morpho_erode(I_comp,B0)

    # Combine the results using logical AND
    return np.logical_and(fg_erode, bg_erode)


In [ ]:
def morpho_thin(I, B, niter=100):
    """
    Perform morphological thinning on a binary image using a set of structuring elements.

    Parameters
    ----------
    I : numpy.ndarray
        The input binary image as a 2D NumPy array.
    B : numpy.ndarray
        The structuring elements used for thinning, as a 3D NumPy array of shape (3, 3, K),
        where K is the number of structuring elements.
    niter : int, optional
        The number of iterations to perform (default is 100).

    Returns
    -------
    numpy.ndarray
        The thinned binary image as a 2D NumPy array.
    """
    # YOUR CODE HERE
    M,N = I.shape
    p, q, K = B.shape
    # initialize output image
    im_out = I.astype(bool)
    # padded image
    # im_pad = np.pad(I,((p//2, p//2),(q//2,q//2)))
    # loop through interations
    print("Working on iteration ", end="")
    for iteration in range(niter):
        print(iteration+1,end=",")
        prev_im = im_out.copy()
        # perform thinning operation with each SE
        for k in range(K):
            #H = hit_or_miss(im_out,B[:,:,k])
            # implementing scipy 
            B1 = B[:,:,k] == 1 #foreground elements
            B0 = B[:,:,k] == 0 #background elements
            H = binary_hit_or_miss(
                im_out, structure1=B1,structure2=B0
            )
            #im_out = im_out - H
            im_out = np.logical_and(im_out,np.logical_not(H))
        # check to see if another iteration is needed
        if np.array_equal(prev_im, im_out):
            print("No improvement found")
            break
    return im_out

##################
# TESTING
B = np.zeros((3, 3, 8))

B[:, :, 0] = [[0, 0, 0], [-1, 1, -1], [1, 1, 1]]
B[:, :, 1] = [[-1, 0, 0], [1, 1, 0], [1, 1, -1]]
B[:, :, 2] = [[1, -1, 0], [1, 1, 0], [1, -1, 0]]
B[:, :, 3] = [[1, 1, -1], [1, 1, 0], [-1, 0, 0]]
B[:, :, 4] = [[1, 1, 1], [-1, 1, -1], [0, 0, 0]]
B[:, :, 5] = [[-1, 1, 1], [0, 1, 1], [0, 0, -1]]
B[:, :, 6] = [[0, -1, 1], [0, 1, 1], [0, -1, 1]]
B[:, :, 7] = [[0, 0, -1], [0, 1, 1], [-1, 1, 1]]


im = np.zeros((100, 150))

im = imageio.v2.imread("UTK_inv.tif").astype(bool)
i_val = 20
im_thin = morpho_thin(im, B, i_val)
im_thin_sol = morphological_processing_sol.morpho_thin(im, B, i_val)

plt.figure(figsize=(10, 15))
plt.subplot(131)
plt.imshow(im, cmap="gray")

plt.subplot(132)
plt.imshow(im_thin, cmap="gray")

assert diff_check(im_thin, im_thin_sol)

In [ ]:
B = np.zeros((3, 3, 8))

B[:, :, 0] = [[0, 0, 0], [-1, 1, -1], [1, 1, 1]]
B[:, :, 1] = [[-1, 0, 0], [1, 1, 0], [1, 1, -1]]
B[:, :, 2] = [[1, -1, 0], [1, 1, 0], [1, -1, 0]]
B[:, :, 3] = [[1, 1, -1], [1, 1, 0], [-1, 0, 0]]
B[:, :, 4] = [[1, 1, 1], [-1, 1, -1], [0, 0, 0]]
B[:, :, 5] = [[-1, 1, 1], [0, 1, 1], [0, 0, -1]]
B[:, :, 6] = [[0, -1, 1], [0, 1, 1], [0, -1, 1]]
B[:, :, 7] = [[0, 0, -1], [0, 1, 1], [-1, 1, 1]]


im = np.zeros((100, 150))

im = imageio.v2.imread("UTK_inv.tif").astype(bool)

im_thin = morpho_thin(im, B, 500)
im_thin_sol = morphological_processing_sol.morpho_thin(im, B, 500)

plt.figure(figsize=(10, 15))
plt.subplot(131)
plt.imshow(im, cmap="gray")

plt.subplot(132)
plt.imshow(im_thin, cmap="gray")

assert diff_check(im_thin, im_thin_sol)

## Pruning

Write a function `morpho_prun(I, numthin, numdil)` to return a pruned foreground of a binary image `I`.  The parameter `numthin` is the number of times the thinning algorithm is applied to delete the end points.  The parameter `numdil` is the number of dilations.  Typically, `numdil` $\leq$ `numthin`.

In [ ]:
def morpho_prun(I, numthin, numdil):
    """
    Perform morphological pruning on a binary image.

    Parameters
    ----------
    I : numpy.ndarray
        The input binary image as a 2D NumPy array.
    numthin : int
        The number of thinning iterations to perform.
    numdil : int
        The number of dilation iterations to perform after thinning.

    Returns
    -------
    numpy.ndarray
        The pruned binary image as a 2D NumPy array.
    """

    # YOUR CODE HERE
    X1 = I.astype(bool).copy()
    H = np.ones((3,3))
    # morphological pruning elements
    B = np.zeros((3,3,8))
    B[:, :, 0] = [[-1, 0, 0], [1, 1, 0], [-1, 0, 0]]
    B[:, :, 1] = [[-1, 1, -1], [0, 1, 0], [0,0,0]]
    B[:, :, 2] = [[0,0,-1], [0,1,1], [0,0,-1]]
    B[:, :, 3] = [[0,0,0], [0, 1, 0], [-1,1,-1]]
    B[:, :, 4] = [[1,0,0], [0, 1, 0], [0, 0, 0]]
    B[:, :, 5] = [[0,0,1], [0, 1, 0], [0, 0, 0]]
    B[:, :, 6] = [[0,0,0], [0, 1, 0], [0, 0,1]]
    B[:, :, 7] = [[0,0,0], [0, 1, 0], [1, 0, 0]]
    # for i in range(8):
    #     print(B[:,:,i])
    # identify endpoints
    for i in range(numthin):
        X2 = np.zeros_like(X1, dtype=bool)
        for k in range(8):
            B1 = B[:,:,k] == 1 #foreground elements
            B0 = B[:,:,k] == 0 #background elements
            endpoints = binary_hit_or_miss(
                X1, structure1=B1,structure2=B0
            )
            X2 = np.logical_or(X2,endpoints)
        ##########################
        # checking endpoints
        # plt.figure()
        # plt.imshow(X2, cmap='gray')
        # plt.title("Detected Endpoints")
        # plt.show()
    
        # remove the endpoints
        X1 = np.logical_and(X1, np.logical_not(X2))
    # dilate the pruned image
    #X1 = binary_dilation(X1, structure=H, iterations=numdil)
    return X1

################3
# testing
im = imageio.v2.imread("UTK_inv.tif").astype(bool)

B = np.zeros((3, 3, 8))

B[:, :, 0] = [[0, 0, 0], [-1, 1, -1], [1, 1, 1]]
B[:, :, 1] = [[-1, 0, 0], [1, 1, 0], [1, 1, -1]]
B[:, :, 2] = [[1, -1, 0], [1, 1, 0], [1, -1, 0]]
B[:, :, 3] = [[1, 1, -1], [1, 1, 0], [-1, 0, 0]]
B[:, :, 4] = [[1, 1, 1], [-1, 1, -1], [0, 0, 0]]
B[:, :, 5] = [[-1, 1, 1], [0, 1, 1], [0, 0, -1]]
B[:, :, 6] = [[0, -1, 1], [0, 1, 1], [0, -1, 1]]
B[:, :, 7] = [[0, 0, -1], [0, 1, 1], [-1, 1, 1]]

p1 = 20
p2 = 5
im_thin = morpho_thin(im, B, 20)
im_pruned = morpho_prun(im_thin, 10, 1)
im_p1 = morpho_prun(im_thin, p1, 1)
im_p2 = morpho_prun(im_thin, p2, 1)

plt.figure(figsize=(10, 10))

plt.subplot(2,2,1)
plt.imshow(im_p1, cmap="gray")
plt.title(f"Prune factor of {p1}")

plt.subplot(2,2,2)
plt.imshow(im_p2, cmap="gray")
plt.title(f"Prune factor of {p2}")

plt.subplot(2,2,3)
plt.imshow(im_pruned, cmap="gray")
plt.title("Pruned")


im_pruned_sol = morphological_processing_sol.morpho_prun(im_thin, 10, 1)

plt.subplot(2,2,4)
plt.imshow(im_pruned_sol, cmap="gray")
plt.title("Pruned Sol")



In [ ]:
im = imageio.v2.imread("UTK_inv.tif").astype(bool)

B = np.zeros((3, 3, 8))

B[:, :, 0] = [[0, 0, 0], [-1, 1, -1], [1, 1, 1]]
B[:, :, 1] = [[-1, 0, 0], [1, 1, 0], [1, 1, -1]]
B[:, :, 2] = [[1, -1, 0], [1, 1, 0], [1, -1, 0]]
B[:, :, 3] = [[1, 1, -1], [1, 1, 0], [-1, 0, 0]]
B[:, :, 4] = [[1, 1, 1], [-1, 1, -1], [0, 0, 0]]
B[:, :, 5] = [[-1, 1, 1], [0, 1, 1], [0, 0, -1]]
B[:, :, 6] = [[0, -1, 1], [0, 1, 1], [0, -1, 1]]
B[:, :, 7] = [[0, 0, -1], [0, 1, 1], [-1, 1, 1]]

im_thin = morpho_thin(im, B, 20)
im_pruned = morpho_prun(im_thin, 10, 1)

plt.figure(figsize=(10, 15))

plt.subplot(131)
plt.imshow(im, cmap="gray")
plt.title("Original")

plt.subplot(132)
plt.imshow(im_thin, cmap="gray")
plt.title("Thinned")

plt.subplot(133)
plt.imshow(im_pruned, cmap="gray")
plt.title("Pruned")

im_pruned_sol = morphological_processing_sol.morpho_prun(im_thin, 10, 1)

assert diff_check(im_pruned, im_pruned_sol)